In [ ]:
!pip install langchain>=1.0.0
!pip install langchain-core>=1.0.0
!pip install langchain-openai
!pip install langchain langchain-anthropic langgraph
!pip install tiktoken
!pip install langchain-community

In [ ]:
!pip install unstructured
!pip install chromadb
!pip install -U sentence-transformers
!pip install langchain-classic

In [ ]:
from langchain_openai import ChatOpenAI
import os
import openai
from google.colab import userdata

# API 키 설정하기.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# ChatOpenAI 모델 초기화하기.
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.5
)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma


In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A,B):
  return dot(A,B) / (norm(A)*norm(B))

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embeddings_model = HuggingFaceBgeEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device' : 'cpu'},
    encode_kwargs={'normalize_embeddings':True},
    )

In [ ]:
loader = DirectoryLoader(path='/content/sample_data', glob = '*.txt', loader_cls = TextLoader)
#loader = TextLoader('/content/sample_data/dummy_data.txt')
data = loader.load()


for d in data:
  print(d.metadata)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 50,
    length_function = len,
)

docs  = text_splitter.split_documents(data)

In [ ]:
# 유사도 직접 확인하기 위한 추가 코드
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 50,
    length_function = len,
)

docs  = text_splitter.split_documents(data)

docs_content = list()

for i in range(len(docs)):
  print(docs[i].page_content + "\n-------------------------\n")
  docs_content.append(docs[i].page_content)

embeddings = embeddings_model.embed_documents(docs_content)

embedded_query = embeddings_model.embed_query("알레르기성 비염에 효과 있는 약은 뭐예요?")
# "알레르기성 비염에 효과 있는 약은 뭐예요? -> 세노바퀵연질캡슐 포함된 chunk의 유사도가 가장 높게 나옴."
# "알레르기에 효과 있는 약은 뭐예요? -> 타이레놀 포함된 chunk의 유사도가 더 높게 나옴."
# 즉, 효능에 적혀 있는 단어 자체가 아니면 제대로 유사도를 판단하지 못함.
for embedding in embeddings:
  print(cos_sim(embedding, embedded_query))

In [ ]:
vectorstore = Chroma.from_documents(
    docs,
    embeddings_model,
    collection_name = 'medicines',
    persist_directory = './db/chromadb',
    collection_metadata = {'hnsw:space' : 'cosine'},
)

In [ ]:
from collections import Counter
from langchain_classic.retrievers import MultiQueryRetriever

query = "두통이 있어. 타이레놀을 먹으면 돼?"
# 제대로 된 답변

query2 = "감기가 있어. 어떤 약을 먹으면 돼?"
# 두통/감기 등 정확한 용어가 아닌 기침을 언급하면 약을 찾지 못함.

query3 = "알레르기 증상이 있어. 어떤 약을 먹으면 돼?"
# 알레르기 / 알러지 / 알레르리성으로 못 찾음.
# 알레르기성 비염 / 결막염 등 정확한 증상을 설명해야 함.

retriever = vectorstore.as_retriever(
    search_kwargs={"k" : 5}
)


template = ''' 다음 context에는 약의 정보인 "효능", "이름"이 포함되어 있습니다.

질문에 주어진 증상과 관련된 "효능"을 가진 약을 찾으세요.

반드시 다음 순서로 생각하세요:
1. 질문에서 증상(예: 두통, 감기 등)을 파악한다.
2. context의 "효능"에서 증상을 찾는다.
3. 그 효능을 가진 약의 "이름"을 찾는다.
4. 만약, 약의 이름을 찾지 못한다면 다른 약을 추천하지 않기.


답변의 조건
- 언급한 증상과 효과적인 약을 언급하기.
- "context" 단어를 포함시키지 않기.
- 그 약이 가진 부가적은 효능, 성분, 특징 등을 부가적으로 추가 설명하기.


context
: {context}

질문
: {question}

답변:
'''

#template2와 search_prompt는 일단 작성함. (사용하지는 않았음.)
template2 = '''
 다음 단어와 유사한 의미를 가지는 단어 10개를 찾으세요 : {word}

'''

prompt = ChatPromptTemplate.from_template(template)
search_prompt = ChatPromptTemplate.from_template(template2)


# 검색한 문서 결과를 한 개의 문단으로 합침.
def format_docs(document):
  return '\n\n'.join(doc.page_content for doc in document)


def get_filtered_docs(x):
  query = x["question"]

  most_docs = vectorstore.similarity_search(query, k =3)
  sources = [d.metadata["source"] for d in most_docs]
  source_file = Counter(sources).most_common(1)[0][0]

  filtered_docs = vectorstore.similarity_search(
      query,
      k=10,
      filter={"source" : source_file}
  )

  return format_docs(filtered_docs)


rag_chain = (
    {"context" : RunnableLambda(get_filtered_docs), "question" : RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

response = rag_chain.invoke({"question" : query3})
print(response)

In [ ]:
# MultiQueryRetriever 코드
from langchain_classic.retrievers import MultiQueryRetriever


# query3으로 "세노바퀵연질캡슐"이 답변되기는 함.
# 이전에 rag_chain만 사용할 때에는 아예 "세노바퀵연질캡슐"이 답변되지 않음.
# 하지만, 실행에 따라서 매번 결과가 바뀜.
# 타이레놀이 나오거나 추천할 약이 없다고 답변할 때도 있음.
# 콧물 + 기침으로는 감기를 유추하지 못함. (알레르기 -> 알레르기성 등으로는 가능)

query4 = "콧물이랑 기침이 나와. 어떤 약을 먹어야 해?"

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = retriever,
    llm = llm
)

relevant_docs = multiquery_retriever.invoke(query3)

context = format_docs(relevant_docs)

response = rag_chain.invoke({"context" : context, "question" : query3})

print(response)